# Reverse String Training Pipeline

A minimal 3-step pipeline to train an LLM to reverse strings:

1. **Create env** - Define `ReverseEnv` with a binary exact-match reward
2. **Create synthetic data** - Generate 50 random strings with their reversals
3. **Run training job** - Upload and launch the training experiment

## Setup

Install dependencies and configure API credentials.

In [ ]:
# Local development setup
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

repo_root = Path.cwd().parent
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [ ]:
# Configuration
# create an API key from the cgft console (https://app.cgft.io/account/api-keys).
API_KEY = "sk_IYfzBWETfgLrxzXLcXfDNvruSnlNLOGHxnMrlhDrOtczzpBdyTMjYeKPcZEMFkNS"
BASE_URL = "https://app.cgft.io"

## Step 1: Create Environment

Define a `ReverseEnv` that rewards the model for correctly reversing an input string. No tools needed — purely a cognitive task.

In [ ]:
import re
from pathlib import Path
from typing import Any

from benchmax.envs.base_env import BaseEnv
from benchmax.envs.types import StandardizedExample, ToolDefinition


SYSTEM_PROMPT = """You are given a string and must reverse it exactly.
Write the reversed string within the xml tags <answer></answer>.
For example, if the input is "hello", your answer should be <answer>olleh</answer>."""


async def exact_match_reward(completion: str, ground_truth: str, **kwargs: Any) -> float:
    """Return 1.0 if the <answer> tag content exactly matches the reversed string, else 0.0."""
    if isinstance(completion, list):
        parts = [
            msg.get("content", "")
            for msg in completion
            if isinstance(msg, dict) and msg.get("role") == "assistant"
        ]
        completion = " ".join(parts)

    match = re.search(r"<answer>(.*?)</answer>", completion, re.DOTALL)
    if not match:
        return 0.0
    return 1.0 if match.group(1).strip() == ground_truth else 0.0


class ReverseEnv(BaseEnv):
    """Environment for testing LLM string reversal — no tools, binary reward."""

    system_prompt: str = SYSTEM_PROMPT

    def __init__(self, dataset_path: str | None = None, **kwargs):
        self._dataset_path = dataset_path

    def get_train_val_split(self, train_ratio: float = 0.7, seed: int = 42, **kwargs):
        from datasets import load_dataset as hf_load_dataset

        ds = hf_load_dataset(
            "json", data_files=str(Path(self._dataset_path).expanduser().absolute())
        )
        dataset = ds["train"].shuffle(seed=seed)
        split_idx = max(1, min(int(len(dataset) * train_ratio), len(dataset) - 1))
        return dataset.select(range(split_idx)), dataset.select(range(split_idx, len(dataset)))

    @classmethod
    def dataset_preprocess(cls, example: Any, **kwargs) -> StandardizedExample:
        return StandardizedExample(
            prompt=example.get("question", ""),
            ground_truth=example.get("answer", ""),
            init_rollout_args={},
        )

    async def list_tools(self) -> list[ToolDefinition]:
        return []

    async def run_tool(self, rollout_id: str, tool_name: str, **tool_args) -> Any:
        return "Error: ReverseEnv has no tools"

    async def compute_reward(
        self, rollout_id: str, completion: str, ground_truth: Any, **kwargs: Any
    ) -> dict[str, float]:
        return {"exact_match_reward": await exact_match_reward(completion, ground_truth, **kwargs)}

## Step 2: Create Synthetic Data

Generate synthetic prompt-output pairs

In [ ]:
import random
import string

random.seed(42)

def _rand_str(length: int) -> str:
    chars = string.ascii_letters + string.digits
    return "".join(random.choices(chars, k=length))

synthetic_dataset = [
    {
        "question": f"Reverse the following string: {s}",
        "answer": s[::-1],
    }
    for s in (_rand_str(random.randint(5, 20)) for _ in range(50))
]

print(f"Generated {len(synthetic_dataset)} examples")
for ex in synthetic_dataset[:3]:
    print(f"  Q: {ex['question']}")
    print(f"  A: {ex['answer']}")

## Step 3: Train Model

Upload the dataset and bundle the environment for training.

In [ ]:
from synthetic_data_prep.trainer import train

experiment_id = train(
    env_class=ReverseEnv,
    env_args={},
    dataset=synthetic_dataset,
    api_key=API_KEY,
)

print(f"Experiment ID: {experiment_id}")

## Monitoring Training: What to Expect

### Early Training Noise

**Early training**: At the start of a training run, rewards will fluctuate and metrics may be noisy. This is completely normal - the model is still learning basic patterns and its outputs are unstable. Give it some time (usually a few dozen steps) and the signal will clean up and you'll start seeing clear trends.

**Exploration before exploitation**: Ideally, you want to see pass@k climbing first before average reward increases significantly. This means your model is exploring the solution space and learning to solve more prompts (breadth) before it optimizes for higher rewards on those prompts (depth). If average reward shoots up while pass@k stays low, you're likely exploiting a narrow set of easy prompts rather than building robust capabilities.

**Healthy training trajectory**: In a well-configured training run, expect pass@k to increase early as the model learns to solve more distinct prompts. Average reward should follow but lag behind. Eventually both should plateau as the model saturates your training distribution.

**Warning signs**:
- pass@k flat while average reward increases → model is overfitting to a narrow subset of prompts
- both metrics stagnate early → training distribution may be too hard, reward signal too sparse